# Symbolic model gallery

Each model is derived with fpgen in the Wigner representation. The notebook displays the effective Hamiltonian, normal and anomalous diffusion, covariance Jacobian, and a one-dimensional reduced dynamics.

In [ ]:
import sympy as sp
from IPython.display import Math, display
from fpgen import boson_modes, MasterEquation, LindbladChannel, derive_kramers_moyal

a, b, c = boson_modes("a", "b", "c")
omega_a, omega_b, omega_c = sp.symbols("omega_a omega_b omega_c", real=True)
g, g_ab, g_ac, k, chi, epsilon = sp.symbols("g g_ab g_ac k chi epsilon", real=True)
gamma_a, gamma_b, gamma_c, Gamma = sp.symbols("gamma_a gamma_b gamma_c Gamma", real=True)

_DISPLAY_BASES = (r"\alpha", r"\beta", r"\gamma", r"\delta")

def _display_phase_matrix(matrix, variables):
    n = len(variables) // 2
    replacements = {}
    for i in range(n):
        base = _DISPLAY_BASES[i]
        replacements[variables[i]] = sp.Symbol(base, complex=True)
        replacements[variables[n + i]] = sp.Symbol(base + r"^{*}", complex=True)
    return sp.Matrix(matrix).applyfunc(sp.simplify).xreplace(replacements)

def _display_dynamics_expression(expression, dynamics):
    replacements = {
        item.symbol: sp.Symbol(item.latex_name, real=True)
        for item in dynamics.state_spec
    }
    return sp.sympify(expression).xreplace(replacements)

def derive_and_display(master, *, augmented=False):
    km = derive_kramers_moyal(master, representation="wigner")
    fpe = km.truncate(2)
    langevin = fpe.to_langevin()
    H = langevin.hamiltonian if augmented else langevin.physical_hamiltonian
    if H is None:
        raise ValueError("The effective Hamiltonian is unavailable.")
    h_label = r"\mathbf H_{\mathrm{aug}}=" if augmented else r"\mathbf H="
    display(Math(h_label + sp.latex(_display_phase_matrix(H, langevin.variables))))
    display(Math(r"\mathbf D=" + sp.latex(_display_phase_matrix(langevin.normal_correlation, langevin.variables))))
    display(Math(r"\mathbf M=" + sp.latex(_display_phase_matrix(langevin.anomalous_correlation, langevin.variables))))
    return km, fpe, langevin

def display_covariance_jacobian(langevin):
    result = langevin.to_covariance_jacobian()
    # display(Math(r"\mathbf r=" + sp.latex(sp.Matrix(result.coordinates))))
    # display(Math(r"\dot{\mathbf r}=" + sp.latex(result.flow_vector)))
    display(Math(r"\mathbf J(\mathbf R)=" + sp.latex(result.jacobian)))
    return result

def display_augmented_jacobian(dynamics):
    J = dynamics.jacobian().applyfunc(
        lambda expr: _display_dynamics_expression(expr, dynamics)
    )
    display(Math(r"\mathbf J=" + sp.latex(J)))

def display_reduced_dynamics(dynamics, retained_id):
    candidate = next(
        item for item in dynamics.find_linear_reductions(
            retained_dimension=1, equation_partitions="matching"
        )
        if item.retained_ids == (retained_id,)
    )
    plan = dynamics.linear_reduce(candidate=candidate)
    retained = dynamics.state_spec[candidate.retained_indices[0]]
    display(Math(r"\text{Retained order parameter: }\quad q=" + retained.latex_name))
    if len(candidate.eliminated_indices) <= 3:
        reduced = plan.materialize()
        G = _display_dynamics_expression(reduced.reduced_residual[0], dynamics)
        display(Math(r"\dot q=G(q;\mathbf p)=" + sp.latex(G)))
        return reduced
    g_reduced = _display_dynamics_expression(plan.retained_residual[0], dynamics)
    display(Math(
        r"\dot q=\left." + sp.latex(g_reduced)
        + r"\right|_{\mathbf y=\mathbf y_*},\qquad"
        + r"\mathbf A(q;\mathbf p)\mathbf y_*+\mathbf b(q;\mathbf p)=0"
    ))
    return plan

## 1. Degenerate parametric oscillator

The pairing drive requires normal and anomalous second moments. The reduced order parameter is the normal occupation-like moment $R_{aa}$.

In [ ]:
master_dpo = MasterEquation(
    modes=[a],
    hamiltonian=omega_a*a.dag*a + epsilon/2*(a.dag**2 + a**2),
    channels=[LindbladChannel(a, gamma_a)],
)
km_dpo, fpe_dpo, langevin_dpo = derive_and_display(master_dpo, augmented=True)
dynamics_dpo = langevin_dpo.to_augmented_second_moment_dynamics(
    parameters=(omega_a, epsilon, gamma_a)
)
display_augmented_jacobian(dynamics_dpo)
reduction_dpo = display_reduced_dynamics(dynamics_dpo, "r_diag_0")

## 2. Two-mode van der Pol oscillator

Mode $a$ has linear gain and nonlinear two-particle loss; mode $b$ has linear loss. The retained order parameter is $R_{aa}$.

In [ ]:
master_vdp = MasterEquation(
    modes=[a, b],
    hamiltonian=omega_a*a.dag*a + omega_b*b.dag*b + g*(a.dag*b + a*b.dag),
    channels=[LindbladChannel(a.dag, gamma_a), LindbladChannel(a**2, Gamma), LindbladChannel(b, gamma_b)],
)
km_vdp, fpe_vdp, langevin_vdp = derive_and_display(master_vdp)
jacobian_vdp = display_covariance_jacobian(langevin_vdp)
dynamics_vdp = langevin_vdp.to_covariance_dynamics(
    parameters=(omega_a, omega_b, g, gamma_a, gamma_b, Gamma)
)
reduction_vdp = display_reduced_dynamics(dynamics_vdp, "r_diag_0")

## 3. Two-mode pair hopping

The nonlinear exchange is $k(a^{\dagger 2}b^2+b^{\dagger 2}a^2)$. The retained order parameter is $\operatorname{Re}R_{ab}$.

In [ ]:
master_pair = MasterEquation(
    modes=[a, b],
    hamiltonian=(omega_a*a.dag*a + omega_b*b.dag*b + g*(a.dag*b + a*b.dag)
                 + k*(a.dag**2*b**2 + b.dag**2*a**2)),
    channels=[LindbladChannel(a.dag, gamma_a), LindbladChannel(b, gamma_b)],
)
km_pair, fpe_pair, langevin_pair = derive_and_display(master_pair)
jacobian_pair = display_covariance_jacobian(langevin_pair)
dynamics_pair = langevin_pair.to_covariance_dynamics(
    parameters=(omega_a, omega_b, g, k, gamma_a, gamma_b)
)
reduction_pair = display_reduced_dynamics(dynamics_pair, "r_re_0_1")

## 4. Two-mode Kerr system

The Kerr nonlinearity acts on mode $a$. We retain $\operatorname{Im}R_{ab}$, which gives a compact cubic reduced equation.

In [ ]:
master_kerr2 = MasterEquation(
    modes=[a, b],
    hamiltonian=(omega_a*a.dag*a + omega_b*b.dag*b + g*(a.dag*b + a*b.dag)
                 + chi*a.dag**2*a**2),
    channels=[LindbladChannel(a.dag, gamma_a), LindbladChannel(b, gamma_b)],
)
km_kerr2, fpe_kerr2, langevin_kerr2 = derive_and_display(master_kerr2)
jacobian_kerr2 = display_covariance_jacobian(langevin_kerr2)
dynamics_kerr2 = langevin_kerr2.to_covariance_dynamics(
    parameters=(omega_a, omega_b, g, chi, gamma_a, gamma_b)
)
reduction_kerr2 = display_reduced_dynamics(dynamics_kerr2, "r_im_0_1")

## 5. Three-mode Kerr system

Modes $a,b$ are lossy, mode $c$ has gain, and the Kerr term acts on $a$. We retain $R_{aa}$; the remaining eight variables are represented by the exact condensed linear solve instead of expanding a very large rational expression.

In [ ]:
master_kerr3 = MasterEquation(
    modes=[a, b, c],
    hamiltonian=(omega_a*a.dag*a + omega_b*b.dag*b + omega_c*c.dag*c
                 + g_ab*(a.dag*b + a*b.dag) + g_ac*(a.dag*c + a*c.dag)
                 + chi*a.dag**2*a**2),
    channels=[LindbladChannel(a, gamma_a), LindbladChannel(b, gamma_b), LindbladChannel(c.dag, gamma_c)],
)
km_kerr3, fpe_kerr3, langevin_kerr3 = derive_and_display(master_kerr3)
jacobian_kerr3 = display_covariance_jacobian(langevin_kerr3)
dynamics_kerr3 = langevin_kerr3.to_covariance_dynamics(
    parameters=(omega_a, omega_b, omega_c, g_ab, g_ac, chi, gamma_a, gamma_b, gamma_c)
)
reduction_kerr3 = display_reduced_dynamics(dynamics_kerr3, "r_diag_0")

## 6. Two-mode cross-Kerr system

The interaction is $\chi a^\dagger a b^\dagger b$. The retained order parameter is $\operatorname{Re}R_{ab}$.

In [ ]:
master_cross = MasterEquation(
    modes=[a, b],
    hamiltonian=(omega_a*a.dag*a + omega_b*b.dag*b + g*(a.dag*b + a*b.dag)
                 + chi*a.dag*a*b.dag*b),
    channels=[LindbladChannel(a.dag, gamma_a), LindbladChannel(b, gamma_b)],
)
km_cross, fpe_cross, langevin_cross = derive_and_display(master_cross)
jacobian_cross = display_covariance_jacobian(langevin_cross)
dynamics_cross = langevin_cross.to_covariance_dynamics(
    parameters=(omega_a, omega_b, g, chi, gamma_a, gamma_b)
)
reduction_cross = display_reduced_dynamics(dynamics_cross, "r_re_0_1")